# Module 2.1: Chat History & Context Engineering

In Module 1 we built a travel agent and watched chat history vanish
the moment we created a new `AgentSession`. This notebook addresses
that gap in two steps:

1. **Persist chat history** to Cosmos DB so it survives across sessions
2. **Apply context engineering** using the framework’s built-in compaction

By the end you will understand:
- How `CosmosHistoryProvider` saves and loads conversation turns
- How the framework’s `CompactionProvider` manages growing history
- Why even persistent, compacted chat history **is not memory**
- What gap remains that only episodic memory can fill


## Prerequisites

- Everything from Module 1 (Python 3.11+, `az login`, `.env`)
- **Azure Cosmos DB** account with RBAC configured
  → Follow [steps/01_setup_cosmos.md](steps/01_setup_cosmos.md) if not done yet
- `COSMOS_ENDPOINT` added to your `.env` file


In [ ]:
%pip install -q -r ./../requirements.txt

## Step 1: Import the Travel Agent

We extracted the travel agent from Module 1 into `shared/travel_agent.py`.
This gives us the same agent (client, tools, system prompt) without
repeating setup code. Every future module imports from here.


In [ ]:
import sys
import sniffio

sys.path.insert(0, "./..")
sniffio.current_async_library_cvar.set("asyncio")

from shared.travel_agent import create_client, create_travel_agent

client, credential = create_client("../../.env")
travel_agent = create_travel_agent(client)
print("Agent ready with tools: search_flights, search_hotels, get_travel_policy")

## Step 2: The Problem (Quick Recap)

A fresh `AgentSession()` has no history. The agent cannot recall anything
from previous conversations — even if they happened seconds ago.


In [ ]:
from agent_framework import AgentSession

async def prove_amnesia():
    session = AgentSession()
    r = await travel_agent.run("What's my name and where am I going?", session=session)
    print(f"Agent: {r.text}")

await prove_amnesia()

## Step 3: Persist Chat History with Cosmos DB

The Microsoft Agent Framework provides `CosmosHistoryProvider` which
hooks into the session lifecycle:

- **Before each turn**: loads prior messages from Cosmos DB into the session
- **After each turn**: saves new messages back to Cosmos DB

The result: chat history survives across sessions, restarts, and
even different machines — as long as you use the same `session_id`.


In [ ]:
import os
from agent_framework import Agent
from agent_framework_azure_cosmos import CosmosHistoryProvider

COSMOS_ENDPOINT = os.environ["COSMOS_ENDPOINT"]

history = CosmosHistoryProvider(
    endpoint=COSMOS_ENDPOINT,
    database_name="travel-memory",
    container_name="chat-history",
    credential=credential,
)
print(f"History provider ready: {COSMOS_ENDPOINT}")


We wire the history provider into a new agent via `context_providers`.
Provider order matters: the history provider loads messages first,
then any other providers (like compaction) can process them.


In [ ]:
from shared.travel_agent import SYSTEM_PROMPT, search_flights, search_hotels, get_travel_policy

persistent_agent = Agent(
    client=client,
    name="TravelAssistant",
    instructions=SYSTEM_PROMPT,
    tools=[search_flights, search_hotels, get_travel_policy],
    context_providers=[history],
)
print("Persistent agent created with history provider")


## Step 4: A Conversation That Persists

Every message (user and assistant) is now automatically written to Cosmos DB.
We pass an explicit `session_id` — this is the key we’ll use to resume later.


In [ ]:
SESSION_ID = "sarah-booking-001"

async def persistent_booking():
    session = AgentSession(session_id=SESSION_ID)

    turns = [
        "Hi, I'm Sarah Chen. I need flights to New York next week.",
        "I always prefer aisle seats and Marriott hotels.",
        "What hotels are available there?",
    ]
    for msg in turns:
        print(f"User: {msg}")
        r = await persistent_agent.run(msg, session=session)
        print(f"Agent: {r.text}\n")

await persistent_booking()

## Step 5: Coming Back Later

Imagine the user closed their browser and came back the next day.
We create a new `AgentSession` with the same `session_id`. The history
provider loads all prior messages from Cosmos DB before the first turn.


In [ ]:
async def resume_next_day():
    session = AgentSession(session_id=SESSION_ID)

    print("User: What's my name and what seat do I prefer?")
    r = await persistent_agent.run(
        "What's my name and what seat do I prefer?", session=session
    )
    print(f"Agent: {r.text}\n")

    print("User: Which hotel chain do I like?")
    r = await persistent_agent.run("Which hotel chain do I like?", session=session)
    print(f"Agent: {r.text}")

await resume_next_day()

## Step 6: Where Persistent Chat History Falls Short

Chat history now survives across sessions. But persisting raw history
has three scaling problems:

1. **Unbounded token growth** — Every message is loaded every turn
2. **Contradictions accumulate** — Old and new preferences both exist
3. **Cost scales linearly** — More history = more tokens per request


## Step 7: Context Engineering with Built-In Compaction

The Microsoft Agent Framework provides a built-in compaction system.
Instead of writing custom summarization code, we use `CompactionProvider`
with a `SummarizationStrategy`:

- **`SummarizationStrategy`** — Uses the LLM to summarize older message
  groups into a compact paragraph, keeping only the most recent turns verbatim
- **`CompactionProvider`** — A context provider that runs compaction
  automatically before and/or after each agent turn

The framework handles grouping messages, tracking what’s been summarized,
and projecting the compacted view to the model. We just configure it.


In [ ]:
from agent_framework import CompactionProvider, SummarizationStrategy

compaction = CompactionProvider(
    before_strategy=SummarizationStrategy(
        client=client,
        target_count=4,   # keep last 4 non-system messages verbatim
        threshold=2,      # trigger when 6+ non-system messages exist
    ),
    history_source_id=history.source_id,
)
print("Compaction provider ready (summarization, target_count=4)")


In [ ]:
compacted_agent = Agent(
    client=client,
    name="TravelAssistant",
    instructions=SYSTEM_PROMPT,
    tools=[search_flights, search_hotels, get_travel_policy],
    context_providers=[history, compaction],  # order matters!
)
print("Agent created with history + compaction providers")


## Step 8: Run the Compacted Agent

We resume Sarah’s session. The history provider loads all messages from
Cosmos DB, then the compaction provider summarizes older groups before
the model sees them. The agent gets a summary + recent messages instead
of the full transcript.


In [ ]:
async def compacted_conversation():
    session = AgentSession(session_id=SESSION_ID)

    print("User: Remind me — what's my name and hotel preference?")
    r = await compacted_agent.run(
        "Remind me — what's my name and hotel preference?", session=session
    )
    print(f"Agent: {r.text}\n")

    print("User: Can you also search for flights to Chicago?")
    r = await compacted_agent.run(
        "Can you also search for flights to Chicago?", session=session
    )
    print(f"Agent: {r.text}")

await compacted_conversation()

The agent still knows Sarah’s name and preferences — even though older
messages were summarized. The framework preserved key facts in the summary
while reducing the token count sent to the model.


### Context Engineering Approaches: Comparison

| Approach | Token Cost | Retains Early Context | Handles Contradictions | Learns Preferences |
|---|---|---|---|---|
| **Raw history** | Grows forever | ✅ Yes | ❌ No — both versions exist | ❌ No |
| **SlidingWindowStrategy** | Fixed (N groups) | ❌ No — oldest dropped | ❌ No | ❌ No |
| **SummarizationStrategy** | Reduced ~60-80% | ⚠️ Lossy but preserves key facts | ❌ No — summary may include either | ❌ No |
| **ContextWindowCompaction** | Budget-based | ❌ Tool results collapsed first | ❌ No | ❌ No |

Summarization is the best of these — it controls cost while preserving key
facts. But notice the last two columns: **no approach handles contradictions
or learns preferences.** They all manage tokens, not knowledge.


## Step 9: Why Chat History Is Not Memory

Even with perfect compaction, the agent still cannot:

- **Learn** — "Sarah always books Marriott" requires extracting a pattern,
  not storing a transcript
- **Resolve contradictions** — "She said Hilton in Jan, Marriott in March"
  requires choosing the latest, not keeping both
- **Generalize** — "Users from Engineering tend to travel to NYC"
  requires reasoning across sessions, not replaying one
- **Forget selectively** — "Cancel that booking" should update state,
  not just add another message

Chat history (compressed or not) is a **record of conversations**.
Memory is **knowledge extracted from conversations**.

That’s the gap that the next notebook fills.


## Summary

### What We Tried

| Stage | Result |
|---|---|
| Persist raw history to Cosmos DB | ✅ History survives sessions — but grows unboundedly |
| `SummarizationStrategy` compaction | ✅ Reduced tokens, agent still works — but lossy |
| **Conclusion** | Context engineering manages cost, not knowledge |

### Architecture So Far

```mermaid
flowchart LR
  U[User] --> A[Agent + Tools]
  A --> S[AgentSession]
  S <--> C[(Cosmos DB)]
  C -->|loads history| CP[CompactionProvider]
  CP -->|summary + recent| A
  CP -.->|still not memory| X[Cannot learn / resolve / generalize]
```

### Next Step

Module 2.2 introduces **episodic memory** — the agent extracts key events
and facts from conversations, stores them as structured knowledge, and
retrieves only what’s relevant. This is the transition from history to memory.
